# Query EcoDB from Python

Read-only examples against the `eco` schema (PostgreSQL / PostGIS).

Start the database first:

```bash
docker compose up -d
```

Install notebook dependencies once:

```bash
pip install -r notebooks/requirements.txt
```

Default connection (same as `etl/db.py` and the README):

| | |
|---|---|
| Host | `localhost` |
| Port | `5432` |
| Database | `eco_monitoring` |
| User / password | `eco` / `eco` |

Override with `PGHOST`, `PGPORT`, `PGUSER`, `PGPASSWORD`, `PGDATABASE` if needed.

In [ ]:
import os

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 120)

url = URL.create(
    "postgresql+psycopg2",
    username=os.environ.get("PGUSER", "eco"),
    password=os.environ.get("PGPASSWORD", "eco"),
    host=os.environ.get("PGHOST", "localhost"),
    port=int(os.environ.get("PGPORT", "5432")),
    database=os.environ.get("PGDATABASE", "eco_monitoring"),
)
engine = create_engine(url, connect_args={"client_encoding": "UTF8"})


def read_sql(sql: str, **params) -> pd.DataFrame:
    return pd.read_sql(text(sql), engine, params=params or None)


read_sql("SELECT current_database() AS db, current_user AS user, version() AS version")

## Schema objects

Tables and ArcGIS Query Layer views in `eco`.

In [ ]:
read_sql(
    """
    SELECT
        n.nspname AS schema,
        c.relname AS name,
        CASE c.relkind
            WHEN 'r' THEN 'table'
            WHEN 'v' THEN 'view'
            ELSE c.relkind::text
        END AS kind
    FROM pg_class c
    JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname = 'eco'
      AND c.relkind IN ('r', 'v')
    ORDER BY kind, name
    """
)

## Catalogs: sites, points, parameters

In [ ]:
sites = read_sql(
    """
    SELECT site_id, site_code, name_ru, region, is_active
    FROM eco.sites
    ORDER BY site_code
    """
)
sites

In [ ]:
points = read_sql(
    """
    SELECT
        p.point_id,
        p.point_code,
        p.name_ru,
        s.site_code,
        p.media_type,
        ST_Y(p.geom) AS lat,
        ST_X(p.geom) AS lon,
        p.is_active
    FROM eco.monitoring_points p
    JOIN eco.sites s ON s.site_id = p.site_id
    ORDER BY p.media_type, p.point_code
    """
)
points.head(15)

In [ ]:
parameters = read_sql(
    """
    SELECT param_id, param_code, name_ru, name_en, unit, media_type
    FROM eco.parameters
    ORDER BY media_type NULLS LAST, param_code
    """
)
parameters

## Lab results

`eco.vw_results_flat` is one row per lab result (the same layer used in ArcGIS).
Geometry is omitted here; use `ST_X` / `ST_Y` if you need coordinates.

In [ ]:
results = read_sql(
    """
    SELECT
        event_date,
        site_code,
        point_code,
        media_type,
        param_code,
        param_name_ru,
        result_value,
        less_than,
        unit,
        qc_code,
        has_exceedance,
        max_exceedance_ratio
    FROM eco.vw_results_flat
    ORDER BY event_date DESC, point_code, param_code
    LIMIT 20
    """
)
results

In [ ]:
summary = read_sql(
    """
    SELECT
        media_type,
        param_code,
        count(*) AS n_results,
        count(*) FILTER (WHERE less_than) AS n_non_detect,
        count(*) FILTER (WHERE has_exceedance) AS n_exceedance,
        min(event_date) AS first_date,
        max(event_date) AS last_date
    FROM eco.vw_results_flat
    GROUP BY media_type, param_code
    ORDER BY media_type, param_code
    """
)
summary

## Exceedances and latest values

- `vw_exceedances_summary` — one row per result × guideline
- `vw_latest_results` — latest result per (point, parameter); QC codes `D` and `R` are excluded

In [ ]:
exceedances = read_sql(
    """
    SELECT
        event_date,
        point_code,
        param_code,
        param_name_ru,
        result_value,
        unit,
        standard_code,
        limit_value_applied,
        ratio,
        exceedance_pct
    FROM eco.vw_exceedances_summary
    ORDER BY ratio DESC
    """
)
exceedances

In [ ]:
latest = read_sql(
    """
    SELECT
        event_date,
        point_code,
        param_code,
        result_value,
        less_than,
        unit,
        has_exceedance
    FROM eco.vw_latest_results
    WHERE media_type = 'air'
    ORDER BY point_code, param_code
    """
)
latest.head(20)

## Filter by parameter

Change `PARAM` to `CO`, `Dust`, `HCN`, `NO2`, or `SO2`.

In [ ]:
PARAM = "SO2"

series = read_sql(
    """
    SELECT
        event_date,
        point_code,
        result_value,
        less_than,
        unit,
        has_exceedance
    FROM eco.vw_results_flat
    WHERE param_code = :param
    ORDER BY event_date, point_code
    """,
    param=PARAM,
)
series

## Your query

Edit the SQL below. Use `:name` placeholders and pass them as keyword arguments to `read_sql`.

In [ ]:
df = read_sql(
    """
    SELECT *
    FROM eco.vw_results_flat
    WHERE media_type = :media
    LIMIT 10
    """,
    media="air",
)
df